# 04: Member C Wish dataset experiment
Use a GPU runtime in Colab/Kaggle and a checkout of feature/kalana. This notebook audits Member A's fixed splits, trains without opening test loaders, then freezes all four models before evaluation. Synthetic test success is not detector accuracy. Do not Run All through the final evaluation until every model is ready and choices are frozen.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO = Path('/content/deepfake-detection')  # change to your actual checkout
DATA_ROOT = Path('/path/to/RealVsFake')   # same image release as Member A
MEMBER_A_CSV = Path('/path/to/member_a.csv')
DATASET_VERSION = 'SET_EXACT_VERSION_FROM_A'
assert REPO.is_dir(), 'Set REPO to your checkout'
os.chdir(REPO)
def run(*args):
    subprocess.run([sys.executable, *args], check=True)


In [ ]:
run('-m', 'pip', 'install', '-r', 'requirements-member-c.txt')
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
run('-m', 'pytest', '-q')


In [ ]:
assert DATASET_VERSION != 'SET_EXACT_VERSION_FROM_A'
assert DATA_ROOT.is_dir() and MEMBER_A_CSV.is_file()
AUDIT = Path('data/manifests/wish_v1')
if not AUDIT.exists():
    run('-m', 'models.vit.prepare_data', '--manifest', str(MEMBER_A_CSV), '--data-root', str(DATA_ROOT), '--dataset-version', DATASET_VERSION, '--output', str(AUDIT))
audit = json.loads((AUDIT / 'audit.json').read_text())
print(json.dumps(audit, indent=2))
assert audit['near_duplicate_pairs'] == 0, 'Complete documented human review before training; see README.'


## Configure and persist
Before continuing, set data.root and data.manifest in configs/vit.yaml and configs/wish/*.yaml to these paths. Set data_root and manifest in configs/crossgen_harness.yaml too. If candidates were reviewed as false positives, configure the review JSON and replace the guard above with the documented review check. Set results_dir to persistent storage or archive each run folder before the session ends. Select a new run_name for each distinct attempt; do not overwrite runs.


In [ ]:
assert torch.cuda.is_available(), 'Use a GPU runtime for real ViT training'
run('-m', 'models.vit.train', '-c', 'configs/vit.yaml', '--smoke')


In [ ]:
# A/B run their three configurations separately with the same manifest.
run('-m', 'models.vit.train', '-c', 'configs/vit.yaml')
# Resume after session interruption using --resume <run>/checkpoints/last.pt.


In [ ]:
READY_FOR_FINAL_EVALUATION = False  # set True only after receiving all four final runs
assert READY_FOR_FINAL_EVALUATION, 'Confirm all choices/checkpoints are final before opening tests'
run('-m', 'models.vit.evaluate_crossgen', 'freeze', '-c', 'configs/crossgen_harness.yaml')
run('-m', 'models.vit.evaluate_crossgen', 'run', '--device', 'cuda')


In [ ]:
import pandas as pd
FINAL = Path('results/comparison/final')
assert json.loads((FINAL / 'status.json').read_text())['status'] == 'complete'
display(pd.read_csv(FINAL / 'model_comparison.csv'))
display(pd.read_csv(FINAL / 'figures/failure_cases.csv').head(20))
run('-m', 'models.vit.evaluate_crossgen', 'plots', '--predictions', str(FINAL / 'predictions.csv'), '--output', 'results/comparison/regenerated')


## Report and demo
Follow reports/final_report_draft.md and docs/member_c_delivery.md. Explain observed failures, source/pretraining confounds and limitations. Use python -m models.vit.predict for a cropped-face demo. Do not tune on these final results or claim universal detection.
